# Data Manipulation with Pandas

**Table of contents**
1. [Combining data: `concat`, `merge`, `join`](#1-concatenation)
2. [Grouping data: `groupby` and the split–apply–combine pattern](#2-groupby--split-apply-combine)
3. [Reshaping data: `pivot`, `pivot_table`, `melt`, `stack` / `unstack`](#3-reshaping-and-pivot-tables)



## 0. Setup

We'll build a small "company" dataset from scratch: employees, departments, and
quarterly sales records. Using synthetic data means every exercise below has a
deterministic, checkable answer.


In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 20)
pd.set_option("display.width", 100)

rng = np.random.default_rng(42)
print("pandas version:", pd.__version__)


In [ ]:
# Departments lookup table 
departments = pd.DataFrame({
    "dept_id": [10, 20, 30, 40],
    "dept_name": ["Engineering", "Sales", "Marketing", "HR"],
    "location": ["Paris", "Lyon", "Paris", "Lille"],
})
departments


In [ ]:
# Employees table
employees = pd.DataFrame({
    "emp_id": range(1, 11),
    "name": ["Alice", "Bob", "Chloe", "David", "Emma",
             "Farid", "Grace", "Hugo", "Ines", "Jack"],
    "dept_id": [10, 10, 20, 20, 30, 30, 10, 40, 20, 99],  # 99 = no matching dept on purpose!
    "salary": [55000, 48000, 62000, 51000, 47000,
               53000, 59000, 45000, 61000, 50000],
    "hire_date": pd.to_datetime([
        "2019-03-01", "2020-07-15", "2018-01-10", "2021-05-20", "2022-02-11",
        "2019-11-30", "2020-09-01", "2023-01-05", "2017-06-17", "2021-08-08"
    ]),
})
employees


In [ ]:
# Quarterly sales tables
sales_q1 = pd.DataFrame({
    "order_id": range(1, 6),
    "emp_id": [3, 4, 9, 3, 4],
    "product": ["Widget", "Gadget", "Widget", "Gizmo", "Widget"],
    "quantity": [10, 5, 8, 3, 12],
    "unit_price": [9.99, 19.99, 9.99, 14.99, 9.99],
    "quarter": "Q1",
})

sales_q2 = pd.DataFrame({
    "order_id": range(6, 12),
    "emp_id": [3, 4, 9, 9, 3, 2],
    "product": ["Widget", "Gadget", "Gizmo", "Widget", "Gadget", "Widget"],
    "quantity": [7, 6, 4, 9, 11, 2],
    "unit_price": [9.99, 19.99, 14.99, 9.99, 19.99, 9.99],
    "quarter": "Q2",
})

sales_q3 = pd.DataFrame({
    "order_id": range(12, 17),
    "emp_id": [4, 3, 9, 2, 3],
    "product": ["Gadget", "Widget", "Gizmo", "Widget", "Gizmo"],
    "quantity": [8, 15, 5, 6, 2],
    "unit_price": [19.99, 9.99, 14.99, 9.99, 14.99],
    "quarter": "Q3",
})

sales_q1


## 1. Concatenation 
#### 1.1 `pd.concat`

Docs: https://pandas.pydata.org/docs/user_guide/merging.html#concatenating-objects

`pd.concat` glues objects together **along an axis**:
- `axis=0` (default): stack rows on top of each other (like appending)
- `axis=1`: stack columns side by side

Key parameters:
- `join="outer"` (default, keep all columns/index) vs `join="inner"` (keep only shared ones)
- `keys=[...]` to create a hierarchical index that tracks the origin of each chunk
- `ignore_index=True` to renumber the resulting index


In [ ]:
# Stack the three quarterly sales tables into one long table (axis=0, the default)
all_sales = pd.concat([sales_q1, sales_q2, sales_q3], ignore_index=True)
all_sales


In [ ]:
# Using `keys` lets us keep track of which quarter each block came from,
# even though we already have a 'quarter' column — useful when you DON'T have one.
tagged = pd.concat([sales_q1, sales_q2, sales_q3], keys=["Q1", "Q2", "Q3"])
tagged.head(7)


In [ ]:
# axis=1: concatenate side-by-side (aligns on the index!)
bonus = pd.Series([1000, 500, 800, 200, 1200, 300, 900, 100, 700, 400],
                   index=employees["emp_id"], name="bonus")
# align employees (indexed by emp_id) with the bonus series
emp_indexed = employees.set_index("emp_id")
pd.concat([emp_indexed, bonus], axis=1).head()


In [ ]:
# join='inner' vs 'outer' when columns differ
df_a = pd.DataFrame({"x": [1, 2], "y": [3, 4]})
df_b = pd.DataFrame({"y": [5, 6], "z": [7, 8]})

print("outer (default) -> keeps all columns, fills gaps with NaN:")
print(pd.concat([df_a, df_b], join="outer", ignore_index=True))

print("\ninner -> keeps only columns present in BOTH:")
print(pd.concat([df_a, df_b], join="inner", ignore_index=True))


### Exercises — Concatenation

**Exercise 1.1.** Concatenate `sales_q1` and `sales_q2` only (not Q3), reset the index,
and store the result in `h1_sales` (H1 = first half of the year).


In [ ]:
# Your code here


In [ ]:
# Solution
h1_sales = pd.concat([sales_q1, sales_q2], ignore_index=True)
h1_sales


**Exercise 1.2.** Create a new one-row DataFrame for a late Q3 order that was missed:
`order_id=17, emp_id=2, product="Gizmo", quantity=4, unit_price=14.99, quarter="Q3"`.
Concatenate it onto `all_sales` (the full year table) and confirm the row count grew by 1.


In [ ]:
# Your code here


In [ ]:
# Solution
late_order = pd.DataFrame([{
    "order_id": 17, "emp_id": 2, "product": "Gizmo",
    "quantity": 4, "unit_price": 14.99, "quarter": "Q3",
}])
before = len(all_sales)
all_sales_fixed = pd.concat([all_sales, late_order], ignore_index=True)
print("rows before:", before, "-> rows after:", len(all_sales_fixed))
all_sales_fixed.tail()


**Exercise 1.3.** Using `keys=["Q1", "Q2", "Q3"]`, build a concatenated DataFrame
*without* `ignore_index`, then use `.loc["Q2"]` to pull out only the Q2 rows from the
combined object. Does the result match `sales_q2`'s data?


In [ ]:
# Your code here


In [ ]:
# Solution
combined = pd.concat([sales_q1, sales_q2, sales_q3], keys=["Q1", "Q2", "Q3"])
combined.loc["Q2"]


**Exercise 1.4 (challenge).** `df_a` has columns `x, y`; `df_b` has columns `y, z`.
Concatenate them with `axis=1` instead of the default `axis=0`. What happens to the
shape and the column names? Why might this be dangerous if the two DataFrames don't
share the same meaning for their index?


In [ ]:
# Your code here


In [ ]:
# Solution
result = pd.concat([df_a, df_b], axis=1)
print(result)
print(result.shape)
# The two frames are aligned by INDEX POSITION LABELS (0,1), not by any business key.
# If df_a and df_b's rows don't actually correspond to the same real-world entity
# at each index label, axis=1 concatenation will silently produce nonsense pairings.
# This is exactly the kind of bug `merge()` (Part 2) is designed to prevent, because
# merge requires you to be explicit about the join key.


#### 1.2. Merge and Join

Docs: https://pandas.pydata.org/docs/user_guide/merging.html#database-style-dataframe-or-named-series-joining-merging

`pd.merge()` combines DataFrames based on the values of one or more **key columns**,
just like a SQL JOIN. `.join()` is a convenience method for merging on the **index**.

| how       | keeps                                            |
|-----------|---------------------------------------------------|
| `inner`   | only keys present in **both** frames (default)     |
| `left`    | all keys from the left frame                       |
| `right`   | all keys from the right frame                       |
| `outer`   | all keys from **either** frame                       |
| `cross`   | cartesian product of both frames                    |


In [ ]:
# Inner join (default): employees whose dept_id exists in departments
inner = pd.merge(employees, departments, on="dept_id", how="inner")
inner[["name", "dept_id", "dept_name"]]


In [ ]:
# Left join: KEEP all employees, even Jack (dept_id=99) who has no matching department
left = pd.merge(employees, departments, on="dept_id", how="left")
left[["name", "dept_id", "dept_name", "location"]]


In [ ]:
# Notice Jack's dept_name/location are NaN -> use `indicator=True` to see the merge origin
merged_ind = pd.merge(employees, departments, on="dept_id", how="left", indicator=True)
merged_ind[["name", "dept_id", "_merge"]]


In [ ]:
# Outer join: keep everything from both sides (here departments has no unmatched
# rows, so outer looks the same as left in this example -- but that's not
# guaranteed in general)
outer = pd.merge(employees, departments, on="dept_id", how="outer")
outer[["name", "dept_id", "dept_name"]].tail()


In [ ]:
# Merging on multiple keys, and using suffixes when both frames share column names
sales_with_emp = pd.merge(all_sales, employees, on="emp_id", how="left",
                           suffixes=("_sale", "_emp"))
sales_with_emp.head()


In [ ]:
# .join() -- convenience wrapper that merges on the INDEX by default
emp_by_id = employees.set_index("emp_id")
dept_by_id = departments.set_index("dept_id")

# join employees (indexed by emp_id) with a bonus Series aligned on the same index
joined = emp_by_id.join(bonus)  # bonus was defined in Part 1, indexed by emp_id
joined.head()


### Exercises — Merge & Join

**Exercise 2.1.** Perform a `right` join of `employees` and `departments` on `dept_id`.
Compare the row count to the `inner` join above. Which departments (if any) have zero
employees, and how do you know from the result?


In [ ]:
# Your code here


In [ ]:
# Solution
right = pd.merge(employees, departments, on="dept_id", how="right")
print(right[["name", "dept_id", "dept_name"]])
# A department with no employees would show up with NaN in the 'name' / employee columns.
# In this dataset every department has at least one employee, so no NaNs appear --
# try adding a 5th department to `departments` with no matching emp_id to see it happen.


**Exercise 2.2.** Merge `all_sales` with `employees` on `emp_id` using `how="inner"`,
then merge the result with `departments` on `dept_id`. Produce a table with columns
`name, dept_name, product, quantity, unit_price, quarter`.


In [ ]:
# Your code here


In [ ]:
# Solution
full = (
    all_sales
    .merge(employees, on="emp_id", how="inner")
    .merge(departments, on="dept_id", how="inner")
)
full[["name", "dept_name", "product", "quantity", "unit_price", "quarter"]].head()


**Exercise 2.3.** Use `indicator=True` on an `outer` merge of `employees` and
`departments` on `dept_id`. Filter the result to show only the rows where `_merge`
is **not** `"both"`.


In [ ]:
# Your code here


In [ ]:
# Solution
outer_ind = pd.merge(employees, departments, on="dept_id", how="outer", indicator=True)
outer_ind[outer_ind["_merge"] != "both"]


**Exercise 2.4 (challenge).** Compute total **revenue** (`quantity * unit_price`)
per employee **name**, using only `merge` + basic column arithmetic (no `groupby` yet --
that's next section, but see if you can get a per-row revenue column and eyeball the total
for "Alice" — she should have zero sales since she never appears in `all_sales`).


In [ ]:
# Your code here


In [ ]:
# Solution
rev = pd.merge(all_sales, employees, on="emp_id", how="left")
rev["revenue"] = rev["quantity"] * rev["unit_price"]
# Alice's emp_id=1 never appears as a seller in all_sales, so a left-merge FROM
# all_sales can never produce an Alice row. To include her (with zero revenue) we'd
# need to start the merge from `employees` instead:
rev_all_emps = pd.merge(employees, all_sales, on="emp_id", how="left")
rev_all_emps["revenue"] = rev_all_emps["quantity"] * rev_all_emps["unit_price"]
rev_all_emps[rev_all_emps["name"] == "Alice"]


## 2. GroupBy — split, apply, combine

Docs: https://pandas.pydata.org/docs/user_guide/groupby.html

`df.groupby(key)` **splits** the data into groups, lets you **apply** a function to
each group (aggregation, transformation, or filtering), and **combines** the results
back into a single object.


In [ ]:
# Build a rich sales table to group on (product, employee, department, revenue)
sales_full = (
    all_sales
    .merge(employees, on="emp_id", how="left")
    .merge(departments, on="dept_id", how="left")
)
sales_full["revenue"] = sales_full["quantity"] * sales_full["unit_price"]
sales_full.head()


In [ ]:
# Simple aggregation: total revenue per product
sales_full.groupby("product")["revenue"].sum()


In [ ]:
# Group by MULTIPLE keys: revenue per product per quarter
sales_full.groupby(["quarter", "product"])["revenue"].sum()


In [ ]:
# .agg() lets you compute several statistics at once, per column
sales_full.groupby("product")["revenue"].agg(["sum", "mean", "count", "max"])


In [ ]:
# Different aggregations for different columns using a dict
sales_full.groupby("dept_name").agg(
    total_revenue=("revenue", "sum"),
    n_orders=("order_id", "count"),
    avg_quantity=("quantity", "mean"),
)


In [ ]:
# Custom aggregation function
def revenue_range(s):
    return s.max() - s.min()

sales_full.groupby("product")["revenue"].agg(revenue_range)


In [ ]:
# transform(): returns an object the SAME SHAPE as the input -- great for
# adding a group-level statistic back onto every row (e.g. "% of product's total revenue")
sales_full["product_total"] = sales_full.groupby("product")["revenue"].transform("sum")
sales_full["pct_of_product_revenue"] = sales_full["revenue"] / sales_full["product_total"]
sales_full[["product", "revenue", "product_total", "pct_of_product_revenue"]].head()


In [ ]:
# filter(): keep only whole groups that satisfy a condition
# e.g. keep only products whose TOTAL revenue exceeds 300
big_products = sales_full.groupby("product").filter(lambda g: g["revenue"].sum() > 300)
big_products["product"].unique()


In [ ]:
# Iterating over groups directly (rarely needed, but good to know it's possible)
for name, group in sales_full.groupby("quarter"):
    print(name, "-> total revenue:", round(group["revenue"].sum(), 2))


### Exercises — GroupBy

**Exercise 3.1.** Compute the number of orders and total quantity sold, grouped by
`emp_id`, sorted from highest to lowest total quantity.


In [ ]:
# Your code here


In [ ]:
# Solution
by_emp = sales_full.groupby("emp_id").agg(
    n_orders=("order_id", "count"),
    total_quantity=("quantity", "sum"),
).sort_values("total_quantity", ascending=False)
by_emp


**Exercise 3.2.** For each department, find the average revenue per order
AND the name of best-selling product (hint: group by `["dept_name", "product"]`,
sum revenue, then find the max per department).


In [ ]:
# Your code here


In [ ]:
# Solution
avg_rev = sales_full.groupby("dept_name")["revenue"].mean()
print("Average revenue per order, by department:")
print(avg_rev, "\n")

by_dept_product = sales_full.groupby(["dept_name", "product"])["revenue"].sum()
best_product_per_dept = by_dept_product.groupby(level="dept_name").idxmax()
print("Best-selling product per department:")
print(best_product_per_dept)


**Exercise 3.3.** Use `transform` to add a column `pct_of_dept_revenue` showing
what percentage of its DEPARTMENT's total revenue each order represents.


In [ ]:
# Your code here


In [ ]:
# Solution
sales_full["dept_total"] = sales_full.groupby("dept_name")["revenue"].transform("sum")
sales_full["pct_of_dept_revenue"] = sales_full["revenue"] / sales_full["dept_total"]
sales_full[["dept_name", "product", "revenue", "pct_of_dept_revenue"]].head()


**Exercise 3.4 (challenge).** Using `filter`, keep only employees (`emp_id`
groups) whose average order **quantity** is greater than 6. Which employees remain?


In [ ]:
# Your code here


In [ ]:
# Solution
high_qty_emps = sales_full.groupby("emp_id").filter(lambda g: g["quantity"].mean() > 6)
result = high_qty_emps[["emp_id", "name"]].drop_duplicates()
result


## 3. Reshaping and Pivot Tables

Docs: https://pandas.pydata.org/docs/user_guide/reshaping.html

- `pivot()`: reshape data (no aggregation — errors if there are duplicate index/column
  combinations)
- `pivot_table()`: like `pivot`, but **aggregates** when there are duplicates
  (default aggregation is `mean`)
- `melt()`: the inverse of pivot — turn columns into rows ("wide" -> "long")
- `stack()` / `unstack()`: pivot the innermost level of a (possibly hierarchical) index
  or columns


In [ ]:
# pivot_table: total revenue by product (rows) and quarter (columns)
pivot_rev = sales_full.pivot_table(
    index="product", columns="quarter", values="revenue", aggfunc="sum", fill_value=0
)
pivot_rev


In [ ]:
# Add margins (row/column totals) and multiple aggregation functions
pivot_multi = sales_full.pivot_table(
    index="product", columns="quarter", values="revenue",
    aggfunc="sum", fill_value=0, margins=True, margins_name="Total"
)
pivot_multi


In [ ]:
# plain pivot() -- works because (product, quarter) here has no duplicate combos
# after we first aggregate revenue with groupby (pivot itself does NOT aggregate)
agg_first = sales_full.groupby(["product", "quarter"], as_index=False)["revenue"].sum()
agg_first.pivot(index="product", columns="quarter", values="revenue")


In [ ]:
# melt(): wide -> long. Undo the pivot_rev table above.
wide = pivot_rev.reset_index()
long = wide.melt(id_vars="product", var_name="quarter", value_name="revenue")
long.sort_values(["product", "quarter"]).head(8)


In [ ]:
# stack / unstack on a MultiIndex result from groupby
grouped = sales_full.groupby(["dept_name", "quarter"])["revenue"].sum()
print("Original (MultiIndex Series):")
print(grouped, "\n")

unstacked = grouped.unstack("quarter")   # quarter moves from index to columns
print("After unstack('quarter'):")
print(unstacked, "\n")

restacked = unstacked.stack()            # back to the original shape
print("After stack() again:")
print(restacked)


### Exercises — Reshaping

**Exercise 4.1.** Build a `pivot_table` showing **total quantity sold** (not revenue)
with `product` as rows and `quarter` as columns. Fill missing combinations with 0.


In [ ]:
# Your code here


In [ ]:
# Solution
qty_pivot = sales_full.pivot_table(
    index="product", columns="quarter", values="quantity", aggfunc="sum", fill_value=0
)
qty_pivot


**Exercise 4.2.** Build a `pivot_table` of average `revenue` per order,
with `dept_name` as rows and `product` as columns (`aggfunc="mean"`).


In [ ]:
# Your code here


In [ ]:
# Solution
avg_pivot = sales_full.pivot_table(
    index="dept_name", columns="product", values="revenue", aggfunc="mean"
)
avg_pivot


**Exercise 4.3.** Take the pivot table from Exercise 4.1 and `melt()` it back
into a long format with columns `product, quarter, quantity`.


In [ ]:
# Your code here


In [ ]:
# Solution
wide_qty = qty_pivot.reset_index()
long_qty = wide_qty.melt(id_vars="product", var_name="quarter", value_name="quantity")
long_qty.sort_values(["product", "quarter"])


**Exercise 4.4 (challenge).** Starting from `sales_full`, group by
`["dept_name", "product", "quarter"]` and sum `revenue` to get a 3-level MultiIndex
Series. Then `unstack` **two** levels at once (`quarter` and `product`) so you end up
with `dept_name` as the only remaining row index. What do the columns look like?


In [ ]:
# Your code here


In [ ]:
# Solution
deep = sales_full.groupby(["dept_name", "product", "quarter"])["revenue"].sum()
result = deep.unstack(["product", "quarter"])
print(result)
# The columns become a MultiIndex of (product, quarter) pairs -- one column per
# combination that actually occurred in the data. Missing combinations show up as NaN.


---

## 5. Recap Exercises
Using `employees`, `departments`, `sales_q1`, `sales_q2`, and `sales_q3`:

1. Concatenate the three quarterly sales tables into one (`concat`).
2. Merge in employee and department info (`merge`).
3. Compute a `revenue` column.
4. Group by `dept_name` and `quarter` to get total revenue per department per quarter
   (`groupby`).
5. Reshape the result into a table with departments as rows and quarters as columns,
   including a `Total` row and column (`pivot_table` with `margins=True`).

Try building this as a single chained pipeline using method chaining
(`.merge().merge().groupby().pivot_table()` etc.) — it's a very common real-world
pandas pattern!


In [ ]:
# Your code here


In [ ]:
# Solution
capstone = (
    pd.concat([sales_q1, sales_q2, sales_q3], ignore_index=True)
    .merge(employees, on="emp_id", how="left")
    .merge(departments, on="dept_id", how="left")
    .assign(revenue=lambda d: d["quantity"] * d["unit_price"])
    .pivot_table(
        index="dept_name", columns="quarter", values="revenue",
        aggfunc="sum", fill_value=0, margins=True, margins_name="Total"
    )
)
capstone
